# Session 11. Testing and measuring agent quality

**A demo that worked once is not a measurement you can rerun.**

- three levels of proof: deterministic tests, trajectory match, LLM judge
- then a dataset, and the same experiment run twice
- project eval requirements announced today, accepted at session 13

In [ ]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()  # reads .env once; nothing below opens a file


def chat_model(size: str = "cheap", **kwargs):
    """A model object for the configured provider. A dozen lines, copy them once."""
    name = os.environ[f"MODEL_{size.upper()}"]  # ids live in .env, never in code
    secret = os.environ["LLM_API_KEY"]
    if os.getenv("LLM_REASONING_EFFORT"):  # gpt-5.x: tools need reasoning "none"
        kwargs.setdefault("reasoning_effort", os.environ["LLM_REASONING_EFFORT"])
    # Gemini over OpenAI-compat drops the reasoning signature: turn two 400s
    if os.getenv("LLM_PROVIDER", "openai_compat") == "google_genai":
        return init_chat_model(f"google_genai:{name}", api_key=secret, **kwargs)
    return init_chat_model(
        f"openai:{name}", api_key=secret, base_url=os.environ["LLM_BASE_URL"], **kwargs
    )


print("provider:", os.getenv("LLM_PROVIDER", "openai_compat"),
      "| strong:", os.environ["MODEL_STRONG"])

## The agent under test

**A museum information desk. Two dependent tools, one loop.**

- `find_exhibit` maps a topic to a hall id; `visit_info` needs that id
- hand-built graph, session-2 shape: the route is our own function, so it is testable
- the evaluators below consume `result["messages"]`; a `create_agent` project feeds them the same way

In [ ]:
from langchain_core.tools import tool

HALLS = {  # a fake museum: no network, same halls every run
    "impressionism": "hall-3",
    "ancient egypt": "hall-1",
    "dutch masters": "hall-2",
    "modern sculpture": "hall-5",
    "space race": "hall-7",
}
VISITS = {  # what the second tool knows, keyed by the first tool's output
    "hall-1": "Open daily 9:00-17:00. Timed entry, book at the desk.",
    "hall-2": "Open Tue-Sun 10:00-18:00. Free with the standard ticket.",
    "hall-3": "Open Tue-Sun 10:00-18:00. Guided tour at 15:00.",
    "hall-5": "Open Fri-Sun 11:00-19:00. The garden closes at dusk.",
    "hall-7": "Open daily 9:00-20:00. Planetarium ticket sold separately.",
}


@tool
def find_exhibit(query: str) -> str:
    """Find the hall id for an exhibition topic, for example 'impressionism'."""
    needle = query.strip().lower()
    for topic, hall in HALLS.items():
        if needle in topic or topic in needle:
            return hall
    # a miss the model can act on, not an exception
    return f"Unknown topic {query!r}. Current exhibitions: {', '.join(sorted(HALLS))}."


@tool
def visit_info(hall: str) -> str:
    """Return opening hours and ticket notes for a hall id from find_exhibit."""
    if hall not in VISITS:
        return f"No hall {hall!r}. Call find_exhibit first to get a valid id."
    return VISITS[hall]


TOOLS = [find_exhibit, visit_info]
print(find_exhibit.invoke({"query": "impressionism"}))
print(find_exhibit.invoke({"query": "dinosaurs"}))

In [ ]:
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode


def should_continue(state: MessagesState) -> str:
    if state["messages"][-1].tool_calls:  # the whole routing decision
        return "tools"
    return END


def build_graph(model):
    bound = model.bind_tools(TOOLS)  # the seam: any model-shaped object fits

    def call_model(state: MessagesState) -> dict:
        return {"messages": [bound.invoke(state["messages"])]}

    builder = StateGraph(MessagesState)
    builder.add_node("model", call_model)
    builder.add_node("tools", ToolNode(TOOLS))
    builder.add_edge(START, "model")
    builder.add_conditional_edges("model", should_continue, ["tools", END])
    builder.add_edge("tools", "model")
    return builder.compile()


museum = build_graph(chat_model("cheap"))  # the agent stays on the cheap model
print(museum.get_graph().draw_mermaid())

QUESTION = "When can I visit the impressionism exhibition?"
EXPECTED = "Hall 3, open Tue-Sun 10:00-18:00."  # the reference answer for QUESTION

result = museum.invoke(
    {"messages": [{"role": "user", "content": QUESTION}]},
    config={"recursion_limit": 8},
)
print(result["messages"][-1].content)

**Keep `result`. Its message list is this session's raw material.**

- level 1 tests the parts that produced it
- level 2 compares that list against a reference trajectory
- level 3 asks another model what it thinks of the last entry
- every invoke passes an explicit `recursion_limit`: a runaway loop bills a paid endpoint

## Level 1: deterministic tests

**A test that calls a live model is a slow, paid coin flip.**

- scripted models answer from a list: free, instant, same verdict every run
- session 7 tested one tool-free node this way; today, the whole loop
- the stock fake breaks exactly where agents begin: `bind_tools`

In [ ]:
from langchain_core.language_models import FakeListChatModel

fake = FakeListChatModel(responses=["Hall 3 has the impressionists."])
print(fake.invoke("anything").content)  # canned replies, in order

try:  # the reason session 7 stopped at a tool-free node
    fake.bind_tools(TOOLS)
except NotImplementedError as error:
    print("bind_tools:", type(error).__name__, repr(str(error)))  # empty message

In [ ]:
from itertools import cycle

from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage


class ScriptedChatModel(GenericFakeChatModel):
    """Replays scripted AIMessages; bind_tools is a no-op."""

    def bind_tools(self, tools, **kwargs):
        return self  # the script already contains the tool calls


SCRIPT = [  # one museum run, written by hand
    AIMessage(content="", tool_calls=[
        {"name": "find_exhibit", "args": {"query": "impressionism"}, "id": "t1"}]),
    AIMessage(content="", tool_calls=[
        {"name": "visit_info", "args": {"hall": "hall-3"}, "id": "t2"}]),
    AIMessage(content=EXPECTED),
]

replay = build_graph(ScriptedChatModel(messages=cycle(SCRIPT))).invoke(
    {"messages": [{"role": "user", "content": QUESTION}]},
    config={"recursion_limit": 8},
)
print(len(replay["messages"]), "messages, no request sent, nothing billed")
print(replay["messages"][-1].content)

**Four tests, pytest-shaped. They run here; they belong in `tests/`.**

- routes are plain functions over hand-built states: no graph, no model
- a tool's miss case is a unit test, not a demo
- copy these verbatim into your repo and run them under pytest

In [ ]:
def test_route_to_tools():
    asks = AIMessage(content="", tool_calls=[  # a state built by hand
        {"name": "find_exhibit", "args": {"query": "egypt"}, "id": "t3"}])
    assert should_continue({"messages": [asks]}) == "tools"


def test_route_to_end():
    assert should_continue({"messages": [AIMessage(content="All set.")]}) == END


def test_find_exhibit_miss():
    reply = find_exhibit.invoke({"query": "dinosaurs"})
    assert "Unknown topic" in reply and "impressionism" in reply


def test_full_run_scripted():
    graph = build_graph(ScriptedChatModel(messages=cycle(SCRIPT)))
    out = graph.invoke(
        {"messages": [{"role": "user", "content": QUESTION}]},
        config={"recursion_limit": 8},
    )
    assert out["messages"][-1].content == EXPECTED
    assert sum(m.type == "tool" for m in out["messages"]) == 2  # both tools ran


TESTS = [test_route_to_tools, test_route_to_end,
         test_find_exhibit_miss, test_full_run_scripted]
for test in TESTS:
    test()
    print("PASSED", test.__name__)

## The GAIA-style scorer

**Same answer, different spelling, zero points. Unless you normalize.**

- the GAIA benchmark grades free text as normalize-then-exact-match
- numbers drop currency signs and commas; lists split; strings lowercase
- no partial credit by design; we take the scorer idea, not the benchmark

In [ ]:
import re
import string


def normalize_number(text: str) -> float | None:
    cleaned = text.replace("$", "").replace("%", "").replace(",", "").strip()
    try:
        return float(cleaned)
    except ValueError:
        return None  # not a number: fall through to list or string


def normalize_str(text: str) -> str:
    bare = text.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", bare).strip().lower()


def question_scorer(answer: str, reference: str) -> bool:
    if normalize_number(reference) is not None:  # the reference picks the type
        return normalize_number(answer) == normalize_number(reference)
    if any(sep in reference for sep in (",", ";")):
        parts = lambda s: [normalize_str(p) for p in re.split("[,;]", s)]
        return parts(answer) == parts(reference)
    return normalize_str(answer) == normalize_str(reference)


CASES = [("1,250", "1250"), ("Hall 3!", "hall 3"),
         ("mummies; pharaohs", "Mummies, Pharaohs"), ("hall 5", "hall 3")]
for answer, reference in CASES:
    print(f"{answer!r:20} vs {reference!r:20}", "naive:", answer == reference,
          " scored:", question_scorer(answer, reference))

print("live answer scores:", question_scorer(result["messages"][-1].content, EXPECTED))

**The last pair still fails. That one should.**

- prose fails exact match even when the facts inside it are right
- that is the design: a scorer that never forgives never flatters
- short factual answers belong at level 1; free prose needs level 3

## Level 2: trajectory match

**The right answer through the wrong tools is luck, not quality.**

- the trajectory is the message list you already read in every trace
- a reference trajectory is a hand-written list of the calls you expect
- `agentevals` compares the two; it takes `result["messages"]` as-is

In [ ]:
import json

from agentevals.trajectory.match import create_trajectory_match_evaluator


def tool_call_step(name: str, **args) -> dict:
    """One assistant step calling one tool, in OpenAI dict form."""
    return {"role": "assistant", "tool_calls": [
        {"function": {"name": name, "arguments": json.dumps(args)}}]}


REFERENCE = [  # the run we expect, written before running anything
    {"role": "user", "content": QUESTION},
    tool_call_step("find_exhibit", query="impressionism"),
    {"role": "tool", "content": "hall-3"},
    tool_call_step("visit_info", hall="hall-3"),
    {"role": "tool", "content": VISITS["hall-3"]},
    # text differs on purpose: the matcher never reads it
    {"role": "assistant", "content": "Any wording works here."},
]

strict_match = create_trajectory_match_evaluator(trajectory_match_mode="strict")
# the scripted replay keeps this deterministic; result["messages"] plugs in the same
print(strict_match(outputs=replay["messages"], reference_outputs=REFERENCE))

**`score: True`, and wording never entered into it.**

- the matcher compares tool calls and arguments, not text
- the reference's assistant text differs on purpose and still scores
- the same evaluator object is reused inside the experiment below

In [ ]:
polite = [  # the agent double-checks the hours: one extra call
    {"role": "user", "content": "Where are the Dutch masters?"},
    tool_call_step("find_exhibit", query="dutch masters"),
    {"role": "tool", "content": "hall-2"},
    tool_call_step("visit_info", hall="hall-2"),
    {"role": "tool", "content": VISITS["hall-2"]},
    {"role": "assistant", "content": "Hall 2, open Tue-Sun."},
]
minimal = [  # the reference stops after the lookup
    {"role": "user", "content": "Where are the Dutch masters?"},
    tool_call_step("find_exhibit", query="dutch masters"),
    {"role": "tool", "content": "hall-2"},
    {"role": "assistant", "content": "Hall 2."},
]

for mode in ("strict", "unordered", "subset", "superset"):
    verdict = create_trajectory_match_evaluator(trajectory_match_mode=mode)(
        outputs=polite, reference_outputs=minimal)
    print(f"{mode:9} ->", verdict["score"])

In [ ]:
reworded = [minimal[0], tool_call_step("find_exhibit", query="dutch painting"),
            minimal[2], minimal[3]]  # same call, different search wording

exact = create_trajectory_match_evaluator(trajectory_match_mode="strict")
loose = create_trajectory_match_evaluator(
    trajectory_match_mode="strict", tool_args_match_mode="ignore")
per_tool = create_trajectory_match_evaluator(
    trajectory_match_mode="strict",
    tool_args_match_overrides={"find_exhibit": "ignore"},  # relax one tool only
)

for name, evaluator in [("exact", exact), ("ignore", loose), ("override", per_tool)]:
    print(f"{name:9} ->", evaluator(outputs=reworded, reference_outputs=minimal)["score"])

**Which mode when. Arguments are a separate axis.**

- strict: a mandated sequence, the policy check before the action
- unordered: retrieval, where call order is noise
- subset: a scope ceiling, no calls beyond the reference
- superset: a required minimum, at least these calls
- args: `exact` / `ignore` globally, or per-tool overrides, chosen per dataset

## Level 3: the LLM judge

**A judge is a prompt plus a structured-output schema. Nothing more.**

- never grade with the model that wrote the answer: self-preference is real
- the agent runs on `MODEL_CHEAP`; the judge runs on `MODEL_STRONG`
- `openevals` takes any LangChain chat model as the judge

In [ ]:
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT

correctness_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,  # placeholders: inputs, outputs, reference_outputs
    judge=chat_model("strong"),  # never the model that wrote the answer
    feedback_key="correctness",
)
graded = correctness_judge(
    inputs=QUESTION,
    outputs=result["messages"][-1].content,
    reference_outputs=EXPECTED,
)
print(graded)

In [ ]:
from openevals.prompts import CONCISENESS_PROMPT

scaled = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    judge=chat_model("strong"),
    feedback_key="correctness_scaled",
    continuous=True,  # any value in 0..1 instead of a boolean
)
choice = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    judge=chat_model("strong"),
    feedback_key="correctness_choice",
    choices=[0.0, 0.5, 1.0],  # enforced by the schema at the provider, not client-side
)
calibrated = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    judge=chat_model("strong"),
    feedback_key="calibrated",
    few_shot_examples=[{  # graded examples teach the scale better than adjectives
        "inputs": "Where is hall 9?",
        "outputs": "There is no hall 9 in this museum.",
        "score": 1.0,
        "reasoning": "A correct refusal counts as a correct answer.",
    }],
)
concise = create_llm_as_judge(
    prompt=CONCISENESS_PROMPT,  # reference-free: placeholders are inputs and outputs
    judge=chat_model("strong"),
    feedback_key="conciseness",
)

answer = result["messages"][-1].content
for judge in (scaled, choice, calibrated):
    print(judge(inputs=QUESTION, outputs=answer, reference_outputs=EXPECTED))
print(concise(inputs=QUESTION, outputs=answer))  # no reference passed at all

**Judges inherit the model's habits. Three are systematic.**

- self-preference: a model favors its own house style; grade with a different one
- verbosity: longer answers score higher at equal substance
- position: in A/B comparisons the first answer wins ties; anonymize and swap
- read the comment, not only the score; session 13's council shows these live

## The dataset

**One question proves nothing. A dataset is the unit of evaluation.**

- each item: input, expected answer, expected trajectory
- five here; ten of your own is the session-13 requirement
- items are JSON-clean dicts, ready to upload as a hosted dataset later
- the trajectory rides inside `expected_output`, not `metadata`: metadata is copied onto every span

In [ ]:
def desk_item(question: str, answer: str, steps: list) -> dict:
    trajectory = [{"role": "user", "content": question}]
    for name, args, reply in steps:  # each step: tool call, then its result
        trajectory += [tool_call_step(name, **args), {"role": "tool", "content": reply}]
    trajectory.append({"role": "assistant", "content": answer})
    # metadata is copied onto every span, capped at 200 chars: no trajectories
    return {"input": question,
            "expected_output": {"answer": answer, "trajectory": trajectory}}


DATASET = [  # five scenarios; your project needs ten
    desk_item(QUESTION, EXPECTED,
              [("find_exhibit", {"query": "impressionism"}, "hall-3"),
               ("visit_info", {"hall": "hall-3"}, VISITS["hall-3"])]),
    desk_item("Which hall has Ancient Egypt, and when is it open?",
              "Hall 1, open daily 9:00-17:00.",
              [("find_exhibit", {"query": "ancient egypt"}, "hall-1"),
               ("visit_info", {"hall": "hall-1"}, VISITS["hall-1"])]),
    desk_item("Where are the Dutch masters, and is a standard ticket enough?",
              "Hall 2, Tue-Sun 10:00-18:00, standard ticket.",
              [("find_exhibit", {"query": "dutch masters"}, "hall-2"),
               ("visit_info", {"hall": "hall-2"}, VISITS["hall-2"])]),
    desk_item("Can I see the modern sculpture on a Monday?",
              "No: hall 5 opens Fri-Sun 11:00-19:00.",
              [("find_exhibit", {"query": "modern sculpture"}, "hall-5"),
               ("visit_info", {"hall": "hall-5"}, VISITS["hall-5"])]),
    desk_item("Do you have a dinosaur exhibition?",  # the planned miss
              "No dinosaur exhibition; the current topics are listed at the desk.",
              [("find_exhibit", {"query": "dinosaurs"},
                find_exhibit.invoke({"query": "dinosaurs"}))]),
]
print(len(DATASET), "items,",
      sum(len(i["expected_output"]["trajectory"]) for i in DATASET), "reference messages")

## The experiment

**Langfuse ships datasets, experiments and judges. No trajectory matcher.**

- the course's answer: agentevals wrapped as plain evaluator functions inside `run_experiment`
- offline dataset runs here, versus online judges Langfuse runs continuously on live traces — the traces you have since session 7
- the next two cells need the local Langfuse of session 2; `flush()` closes the run

In [ ]:
from langfuse import Evaluation, get_client

langfuse = get_client()  # reads LANGFUSE_HOST and both keys from the environment


def museum_task(*, item, **kwargs):
    run = museum.invoke(
        {"messages": [{"role": "user", "content": item["input"]}]},
        config={"recursion_limit": 8},
    )
    # evaluators only see input/output/expected_output/metadata: pack both in
    return {"answer": run["messages"][-1].content, "messages": run["messages"]}


def exact_match_eval(*, output, expected_output, **kwargs):
    verdict = question_scorer(output["answer"], expected_output["answer"])
    return Evaluation(name="exact_match", value=float(verdict))


def trajectory_eval(*, output, expected_output, **kwargs):
    verdict = strict_match(outputs=output["messages"],
                           reference_outputs=expected_output["trajectory"])
    return Evaluation(name=verdict["key"], value=float(verdict["score"]),
                      comment=verdict["comment"])


def judge_eval(*, input, output, expected_output, **kwargs):
    graded = correctness_judge(inputs=input, outputs=output["answer"],
                               reference_outputs=expected_output["answer"])
    return Evaluation(name=graded["key"], value=graded["score"],
                      comment=graded["comment"])


def accuracy(*, item_results, **kwargs):  # one number for the whole run
    scores = [e.value for r in item_results
              for e in r.evaluations if e.name == "exact_match"]
    return Evaluation(name="accuracy", value=sum(scores) / len(scores))


experiment = langfuse.run_experiment(
    name="museum-desk-baseline",
    data=DATASET,
    task=museum_task,
    evaluators=[exact_match_eval, trajectory_eval, judge_eval],
    run_evaluators=[accuracy],
)
print(experiment.format())
langfuse.flush()  # a notebook kernel never exits; nothing sends without this

In [ ]:
STRICT_PROMPT = "Answer in one short sentence: hall id, days, hours. Nothing else."


def strict_task(*, item, **kwargs):
    run = museum.invoke(
        {"messages": [{"role": "system", "content": STRICT_PROMPT},
                      {"role": "user", "content": item["input"]}]},
        config={"recursion_limit": 8},
    )
    # drop the system turn: the reference trajectories start at the user
    return {"answer": run["messages"][-1].content, "messages": run["messages"][1:]}


comparison = langfuse.run_experiment(
    name="museum-desk-strict-prompt",  # same dataset, new task: comparable runs
    data=DATASET,
    task=strict_task,
    evaluators=[exact_match_eval, trajectory_eval, judge_eval],
    run_evaluators=[accuracy],
)
print(comparison.format())  # both format() tables go into runs/session-11.md
langfuse.flush()

**Two runs over one dataset sit side by side in the UI.**

- the experiment view: per-item scores, run-level accuracy, runs as columns
- this is exactly the practice task: compare two prompts or two models
- every item links to its trace: evaluation and observability share rails

## Reliability and pass^k

**One green run proves little. Quality is a probability.**

- pass^k: the chance that all k independent runs succeed, p to the power k
- a 90-percent agent survives eight runs 43 percent of the time
- this arithmetic is why the requirements below say three runs

In [ ]:
def pass_k(p: float, k: int) -> float:
    return p ** k  # assumes the runs are independent


print("        k=1   k=3   k=8")
for p in (0.9, 0.95, 0.99):
    row = "  ".join(f"{pass_k(p, k):.2f}" for k in (1, 3, 8))
    print(f"p={p:.2f}  {row}")

## tau2-bench, the idea

**Benchmarks agents the way users break them: in conversation.**

- an LLM plays the user, improvising within a written policy
- the agent under test works its tools over a mock database
- the reward reads the database end-state; the metric is pass^k
- full runs cost real money; the ideas transfer for free

## Practice

**Your own project agent, measured. Course assistant as the fallback.**

1. `pip install agentevals openevals` into your project venv
2. a dataset of 10 scenarios: input, expected answer, expected trajectory
3. wire all three levels: pytest with a scripted model, a trajectory match, one LLM judge
4. run the experiment twice, two prompts or two models, compare in the Langfuse UI, and write the one-sentence judge critique the artifact asks for

**Project eval tests — accepted at session 13, at the instructor's desk:**

- evaluation dataset of at least 10 scenarios
- at least 1 deterministic trajectory test
- at least 1 LLM evaluation
- a numeric quality threshold, stated in advance
- results reported over 3 runs

**These numbers are how you defend the project.**

- at the defense, eval results are the argument for the agent's value
- oral-exam questions on this topic are among the hardest in the set

**Required artifact: `runs/session-11.md`, committed.**

- the green pytest line, and the 10-scenario dataset or an in-repo link to it
- `format()` output of both experiment runs
- one judge score with one sentence of your own critique
- an exported trace of one experiment item, committed next to it

**Stretch, if you finish early.**

- a `tool_args_match_overrides` entry your dataset genuinely needs
- a second run-level evaluator: mean judge score, or the worst item

## Not taught today

**Evaluators we point at and do not run.**

- agentevals also ships an LLM trajectory judge and graph-trajectory variants
- memory has its own evals: survey techniques 28-29, benchmarks LoCoMo and LongMemEval; full runs cost real money
- online continuous evaluation: Langfuse can run judges like these on your live traces (the session-7 instrumentation)

## Next time

**How industrial harnesses are built: one main loop, layered context compression, cheap models for small jobs.**